In [1]:
import requests

r = requests.get("https://gamma-api.polymarket.com/events?closed=false")
response = r.json()

response

[{'id': '16084',
  'ticker': 'fed-rate-hike-in-2025',
  'slug': 'fed-rate-hike-in-2025',
  'title': 'Fed rate hike in 2025?',
  'description': 'This market will resolve to “Yes” if the upper bound of the target federal funds rate is increased at any point between January 1, 2025 and the Fed\'s December 2025 meeting, currently scheduled for December 9-10. Otherwise, this market will resolve to “No”.\n\nThis market may not resolve to "No" until the Fed has released its rate changes information following its December meeting.\n\nThe primary resolution source for this market will be the official website of the Federal Reserve (https://www.federalreserve.gov/monetarypolicy/openmarket.htm), however a consensus of credible reporting may also be used.',
  'startDate': '2024-12-29T22:50:44.013518Z',
  'creationDate': '2024-12-29T22:50:44.013516Z',
  'endDate': '2025-12-10T12:00:00Z',
  'image': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/will-the-fed-raise-interest-rates-in-2025-PQTEY

In [55]:
# Create a pandas DataFrame from the Polymarket /events?closed=false response JSON.
# - Keeps ALL original market fields untouched (as columns)
# - Adds event fields with an 'event_' prefix (so you know the parent event)
# - Optionally parses JSON-in-strings and datetimes into helper columns

import pandas as pd
import json

def markets_df_from_events_response(response, parse_helpers=True):
    """
    response: the JSON returned by GET https://gamma-api.polymarket.com/events?closed=false
              (can be a list of events or a dict with an 'events' key)
    returns:  pandas.DataFrame with one row per market
    """
    # unwrap events list
    if isinstance(response, list):
        events = response
    elif isinstance(response, dict):
        events = response.get("events") or response.get("data") or [response]
        if not isinstance(events, list):
            events = [events]
    else:
        raise TypeError("Unsupported response type")

    rows = []
    for ev in events:
        markets = ev.get("markets") or ev.get("marketsData") or []
        # event metadata (prefix to avoid overwriting market keys)
        ev_meta = {f"event_{k}": v for k, v in ev.items() if k not in ("markets", "marketsData")}
        for m in markets:
            row = dict(m)       # keep market fields as-is
            row.update(ev_meta) # attach event info
            rows.append(row)

    df = pd.DataFrame(rows)

    if parse_helpers and not df.empty:
        # parse JSON-in-string arrays into helper columns (keep raw columns untouched)
        json_str_cols = ["outcomes", "shortOutcomes", "outcomePrices", "clobTokenIds", "umaResolutionStatuses"]
        for col in json_str_cols:
            if col in df.columns:
                def _parse_list(x):
                    if isinstance(x, str):
                        s = x.strip()
                        if s.startswith("[") and s.endswith("]"):
                            try:
                                return json.loads(s)
                            except Exception:
                                return None
                    return x
                df[col + "_list"] = df[col].apply(_parse_list)

        # parse datetimes into helper columns (keep raw strings)
        dt_cols = [
            "startDate","endDate","createdAt","updatedAt","acceptingOrdersTimestamp",
            "event_startDate","event_endDate","event_createdAt","event_updatedAt"
        ]
        for col in dt_cols:
            if col in df.columns:
                df[col + "_dt"] = pd.to_datetime(df[col], utc=True, errors="coerce")

        # coerce common numeric fields to numeric helper columns (raw kept)
        num_cols = [
            "volume","volumeNum","volumeClob","volume24hr","volume1wk","volume1mo","volume1yr",
            "volume24hrClob","volume1wkClob","volume1moClob","volume1yrClob",
            "liquidity","liquidityNum","liquidityClob",
            "bestBid","bestAsk","lastTradePrice","spread","orderPriceMinTickSize","orderMinSize",
            "competitive","rewardsMinSize","rewardsMaxSpread"
        ]
        for col in num_cols:
            if col in df.columns:
                df[col + "_num"] = pd.to_numeric(df[col], errors="coerce")

    return df

# --- usage ---
# import requests
# r = requests.get("https://gamma-api.polymarket.com/events?closed=false", timeout=15)
# response = r.json()
# df = markets_df_from_events_response(response)
# df.head()



In [56]:
df = markets_df_from_events_response(response)
df

,id,question,conditionId,slug,resolutionSource,endDate,liquidity,startDate,image,icon,description,outcomes,outcomePrices,volume,active,closed,marketMakerAddress,createdAt,updatedAt,new,featured,submitted_by,archived,resolvedBy,restricted,groupItemTitle,groupItemThreshold,questionID,enableOrderBook,orderPriceMinTickSize,orderMinSize,volumeNum,liquidityNum,endDateIso,startDateIso,hasReviewedDates,volume24hr,volume1wk,volume1mo,volume1yr,clobTokenIds,umaBond,umaReward,volume24hrClob,volume1wkClob,volume1moClob,volume1yrClob,volumeClob,liquidityClob,acceptingOrders,negRisk,ready,funded,acceptingOrdersTimestamp,cyom,competitive,pagerDutyNotificationEnabled,approved,rewardsMinSize,rewardsMaxSpread,spread,oneDayPriceChange,oneWeekPriceChange,oneMonthPriceChange,lastTradePrice,bestBid,bestAsk,automaticallyActive,clearBookOnStart,manualActivation,negRiskOther,umaResolutionStatuses,pendingDeployment,deploying,rfqEnabled,holdingRewardsEnabled,feesEnabled,event_id,event_ticker,event_slug,event_title,event_description,event_startDate,event_creationDate,event_endDate,event_image,event_icon,event_active,event_closed,event_archived,event_new,event_featured,event_restricted,event_liquidity,event_volume,event_openInterest,event_createdAt,event_updatedAt,event_competitive,event_volume24hr,event_volume1wk,event_volume1mo,event_volume1yr,event_enableOrderBook,event_liquidityClob,event_commentCount,event_tags,event_cyom,event_showAllOutcomes,event_showMarketImages,event_enableNegRisk,event_automaticallyActive,event_negRiskAugmented,event_pendingDeployment,event_deploying,closedTime,umaEndDate,umaResolutionStatus,negRiskMarketID,negRiskRequestID,clobRewards,automaticallyResolved,seriesColor,event_resolutionSource,event_negRisk,event_negRiskMarketID,event_gmpChartMode,showGmpSeries,showGmpOutcome,oneHourPriceChange,event_sortBy,event_series,event_seriesSlug,volume24hrAmm,volume1wkAmm,volume1moAmm,volume1yrAmm,volumeAmm,liquidityAmm,oneYearPriceChange,gameStartTime,event_startTime,customLiveness,deployingTimestamp,event_featuredOrder,event_eventDate,outcomes_list,outcomePrices_list,clobTokenIds_list,umaResolutionStatuses_list,startDate_dt,endDate_dt,createdAt_dt,updatedAt_dt,acceptingOrdersTimestamp_dt,event_startDate_dt,event_endDate_dt,event_createdAt_dt,event_updatedAt_dt,volume_num,volumeNum_num,volumeClob_num,volume24hr_num,volume1wk_num,volume1mo_num,volume1yr_num,volume24hrClob_num,volume1wkClob_num,volume1moClob_num,volume1yrClob_num,liquidity_num,liquidityNum_num,liquidityClob_num,bestBid_num,bestAsk_num,lastTradePrice_num,spread_num,orderPriceMinTickSize_num,orderMinSize_num,competitive_num,rewardsMinSize_num,rewardsMaxSpread_num
0,516706,Fed rate hike in 2025?,0x4319532e181605cb15b1bd677759a3bc7f7394b2fdf145195b700e...,fed-rate-hike-in-2025,,2025-12-10T12:00:00Z,60848.22599,2024-12-29T22:50:33.584839Z,https://polymarket-upload.s3.us-east-2.amazonaws.com/wil...,https://polymarket-upload.s3.us-east-2.amazonaws.com/wil...,This market will resolve to “Yes” if the upper bound of ...,"[""Yes"", ""No""]","[""0.0135"", ""0.9865""]",736030.554114,True,False,,2024-12-29T17:38:00.916304Z,2025-11-08T19:52:19.294139Z,False,False,0x91430CaD2d3975766499717fA0D66A78D814E5c5,False,0x6A9D222616C90FcA5754cd1333cFD9b7fb6a4F74,True,,0,0x8428884817cbc26422ec451101fcedfc5995907a8df6e5905bc29c...,True,0.001,5,7.360306e+05,60848.22599,2025-12-10,2024-12-29,True,6078.040000,6.098663e+04,1.851567e+05,7.360106e+05,"[""604871169844680209782472254744886767496010018298867559...",500,5,6078.040000,6.098663e+04,1.851567e+05,7.360106e+05,7.360306e+05,60848.22599,True,False,False,False,2024-12-29T22:49:15Z,False,0.808615,False,True,100,3.5,0.001,-0.001,0.0015,-0.0110,0.015,0.013,0.014,True,True,False,False,[],False,False,False,False,False,16084,fed-rate-hike-in-2025,fed-rate-hike-in-2025,Fed rate hike in 2025?,This market will resolve to “Yes” if the upper bound of ...,2024-12-29T22:50:44.013518Z,2024-12-29T22:50:44.013516Z,2025-12-10T12:00:00Z,https://polymarket-upload.s3.u

In [59]:
df_active = df[df['active'] == True]
df_active

,id,question,conditionId,slug,resolutionSource,endDate,liquidity,startDate,image,icon,description,outcomes,outcomePrices,volume,active,closed,marketMakerAddress,createdAt,updatedAt,new,featured,submitted_by,archived,resolvedBy,restricted,groupItemTitle,groupItemThreshold,questionID,enableOrderBook,orderPriceMinTickSize,orderMinSize,volumeNum,liquidityNum,endDateIso,startDateIso,hasReviewedDates,volume24hr,volume1wk,volume1mo,volume1yr,clobTokenIds,umaBond,umaReward,volume24hrClob,volume1wkClob,volume1moClob,volume1yrClob,volumeClob,liquidityClob,acceptingOrders,negRisk,ready,funded,acceptingOrdersTimestamp,cyom,competitive,pagerDutyNotificationEnabled,approved,rewardsMinSize,rewardsMaxSpread,spread,oneDayPriceChange,oneWeekPriceChange,oneMonthPriceChange,lastTradePrice,bestBid,bestAsk,automaticallyActive,clearBookOnStart,manualActivation,negRiskOther,umaResolutionStatuses,pendingDeployment,deploying,rfqEnabled,holdingRewardsEnabled,feesEnabled,event_id,event_ticker,event_slug,event_title,event_description,event_startDate,event_creationDate,event_endDate,event_image,event_icon,event_active,event_closed,event_archived,event_new,event_featured,event_restricted,event_liquidity,event_volume,event_openInterest,event_createdAt,event_updatedAt,event_competitive,event_volume24hr,event_volume1wk,event_volume1mo,event_volume1yr,event_enableOrderBook,event_liquidityClob,event_commentCount,event_tags,event_cyom,event_showAllOutcomes,event_showMarketImages,event_enableNegRisk,event_automaticallyActive,event_negRiskAugmented,event_pendingDeployment,event_deploying,closedTime,umaEndDate,umaResolutionStatus,negRiskMarketID,negRiskRequestID,clobRewards,automaticallyResolved,seriesColor,event_resolutionSource,event_negRisk,event_negRiskMarketID,event_gmpChartMode,showGmpSeries,showGmpOutcome,oneHourPriceChange,event_sortBy,event_series,event_seriesSlug,volume24hrAmm,volume1wkAmm,volume1moAmm,volume1yrAmm,volumeAmm,liquidityAmm,oneYearPriceChange,gameStartTime,event_startTime,customLiveness,deployingTimestamp,event_featuredOrder,event_eventDate,outcomes_list,outcomePrices_list,clobTokenIds_list,umaResolutionStatuses_list,startDate_dt,endDate_dt,createdAt_dt,updatedAt_dt,acceptingOrdersTimestamp_dt,event_startDate_dt,event_endDate_dt,event_createdAt_dt,event_updatedAt_dt,volume_num,volumeNum_num,volumeClob_num,volume24hr_num,volume1wk_num,volume1mo_num,volume1yr_num,volume24hrClob_num,volume1wkClob_num,volume1moClob_num,volume1yrClob_num,liquidity_num,liquidityNum_num,liquidityClob_num,bestBid_num,bestAsk_num,lastTradePrice_num,spread_num,orderPriceMinTickSize_num,orderMinSize_num,competitive_num,rewardsMinSize_num,rewardsMaxSpread_num
0,516706,Fed rate hike in 2025?,0x4319532e181605cb15b1bd677759a3bc7f7394b2fdf145195b700e...,fed-rate-hike-in-2025,,2025-12-10T12:00:00Z,60848.22599,2024-12-29T22:50:33.584839Z,https://polymarket-upload.s3.us-east-2.amazonaws.com/wil...,https://polymarket-upload.s3.us-east-2.amazonaws.com/wil...,This market will resolve to “Yes” if the upper bound of ...,"[""Yes"", ""No""]","[""0.0135"", ""0.9865""]",736030.554114,True,False,,2024-12-29T17:38:00.916304Z,2025-11-08T19:52:19.294139Z,False,False,0x91430CaD2d3975766499717fA0D66A78D814E5c5,False,0x6A9D222616C90FcA5754cd1333cFD9b7fb6a4F74,True,,0,0x8428884817cbc26422ec451101fcedfc5995907a8df6e5905bc29c...,True,0.001,5,7.360306e+05,60848.22599,2025-12-10,2024-12-29,True,6078.040000,6.098663e+04,1.851567e+05,7.360106e+05,"[""604871169844680209782472254744886767496010018298867559...",500,5,6078.040000,6.098663e+04,1.851567e+05,7.360106e+05,7.360306e+05,60848.22599,True,False,False,False,2024-12-29T22:49:15Z,False,0.808615,False,True,100,3.5,0.001,-0.001,0.0015,-0.0110,0.015,0.013,0.014,True,True,False,False,[],False,False,False,False,False,16084,fed-rate-hike-in-2025,fed-rate-hike-in-2025,Fed rate hike in 2025?,This market will resolve to “Yes” if the upper bound of ...,2024-12-29T22:50:44.013518Z,2024-12-29T22:50:44.013516Z,2025-12-10T12:00:00Z,https://polymarket-upload.s3.u

In [60]:
df_liquid = df_active[pd.to_numeric(df["liquidity"], errors="coerce").notna()]

/var/folders/b0/vnw_t5qn7zdccn_qsc7_4cs40000gn/T/ipykernel_73913/3623040812.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_liquid = df_active[pd.to_numeric(df["liquidity"], errors="coerce").notna()]


In [61]:

df_filtered = df_liquid.dropna(subset=["liquidity"])


In [62]:
df_filtered

,id,question,conditionId,slug,resolutionSource,endDate,liquidity,startDate,image,icon,description,outcomes,outcomePrices,volume,active,closed,marketMakerAddress,createdAt,updatedAt,new,featured,submitted_by,archived,resolvedBy,restricted,groupItemTitle,groupItemThreshold,questionID,enableOrderBook,orderPriceMinTickSize,orderMinSize,volumeNum,liquidityNum,endDateIso,startDateIso,hasReviewedDates,volume24hr,volume1wk,volume1mo,volume1yr,clobTokenIds,umaBond,umaReward,volume24hrClob,volume1wkClob,volume1moClob,volume1yrClob,volumeClob,liquidityClob,acceptingOrders,negRisk,ready,funded,acceptingOrdersTimestamp,cyom,competitive,pagerDutyNotificationEnabled,approved,rewardsMinSize,rewardsMaxSpread,spread,oneDayPriceChange,oneWeekPriceChange,oneMonthPriceChange,lastTradePrice,bestBid,bestAsk,automaticallyActive,clearBookOnStart,manualActivation,negRiskOther,umaResolutionStatuses,pendingDeployment,deploying,rfqEnabled,holdingRewardsEnabled,feesEnabled,event_id,event_ticker,event_slug,event_title,event_description,event_startDate,event_creationDate,event_endDate,event_image,event_icon,event_active,event_closed,event_archived,event_new,event_featured,event_restricted,event_liquidity,event_volume,event_openInterest,event_createdAt,event_updatedAt,event_competitive,event_volume24hr,event_volume1wk,event_volume1mo,event_volume1yr,event_enableOrderBook,event_liquidityClob,event_commentCount,event_tags,event_cyom,event_showAllOutcomes,event_showMarketImages,event_enableNegRisk,event_automaticallyActive,event_negRiskAugmented,event_pendingDeployment,event_deploying,closedTime,umaEndDate,umaResolutionStatus,negRiskMarketID,negRiskRequestID,clobRewards,automaticallyResolved,seriesColor,event_resolutionSource,event_negRisk,event_negRiskMarketID,event_gmpChartMode,showGmpSeries,showGmpOutcome,oneHourPriceChange,event_sortBy,event_series,event_seriesSlug,volume24hrAmm,volume1wkAmm,volume1moAmm,volume1yrAmm,volumeAmm,liquidityAmm,oneYearPriceChange,gameStartTime,event_startTime,customLiveness,deployingTimestamp,event_featuredOrder,event_eventDate,outcomes_list,outcomePrices_list,clobTokenIds_list,umaResolutionStatuses_list,startDate_dt,endDate_dt,createdAt_dt,updatedAt_dt,acceptingOrdersTimestamp_dt,event_startDate_dt,event_endDate_dt,event_createdAt_dt,event_updatedAt_dt,volume_num,volumeNum_num,volumeClob_num,volume24hr_num,volume1wk_num,volume1mo_num,volume1yr_num,volume24hrClob_num,volume1wkClob_num,volume1moClob_num,volume1yrClob_num,liquidity_num,liquidityNum_num,liquidityClob_num,bestBid_num,bestAsk_num,lastTradePrice_num,spread_num,orderPriceMinTickSize_num,orderMinSize_num,competitive_num,rewardsMinSize_num,rewardsMaxSpread_num
0,516706,Fed rate hike in 2025?,0x4319532e181605cb15b1bd677759a3bc7f7394b2fdf145195b700e...,fed-rate-hike-in-2025,,2025-12-10T12:00:00Z,60848.22599,2024-12-29T22:50:33.584839Z,https://polymarket-upload.s3.us-east-2.amazonaws.com/wil...,https://polymarket-upload.s3.us-east-2.amazonaws.com/wil...,This market will resolve to “Yes” if the upper bound of ...,"[""Yes"", ""No""]","[""0.0135"", ""0.9865""]",736030.554114,True,False,,2024-12-29T17:38:00.916304Z,2025-11-08T19:52:19.294139Z,False,False,0x91430CaD2d3975766499717fA0D66A78D814E5c5,False,0x6A9D222616C90FcA5754cd1333cFD9b7fb6a4F74,True,,0,0x8428884817cbc26422ec451101fcedfc5995907a8df6e5905bc29c...,True,0.001,5,7.360306e+05,60848.22599,2025-12-10,2024-12-29,True,6078.040000,6.098663e+04,1.851567e+05,7.360106e+05,"[""604871169844680209782472254744886767496010018298867559...",500,5,6078.040000,6.098663e+04,1.851567e+05,7.360106e+05,7.360306e+05,60848.22599,True,False,False,False,2024-12-29T22:49:15Z,False,0.808615,False,True,100,3.5,0.001,-0.0010,0.0015,-0.0110,0.015,0.013,0.014,True,True,False,False,[],False,False,False,False,False,16084,fed-rate-hike-in-2025,fed-rate-hike-in-2025,Fed rate hike in 2025?,This market will resolve to “Yes” if the upper bound of ...,2024-12-29T22:50:44.013518Z,2024-12-29T22:50:44.013516Z,2025-12-10T12:00:00Z,https://polymarket-upload.s3.

In [70]:
df = df_filtered

In [72]:
df_sample = df.sample(n=10, random_state=42)

In [73]:
df_sample

,id,question,conditionId,slug,resolutionSource,endDate,liquidity,startDate,image,icon,description,outcomes,outcomePrices,volume,active,closed,marketMakerAddress,createdAt,updatedAt,new,featured,submitted_by,archived,resolvedBy,restricted,groupItemTitle,groupItemThreshold,questionID,enableOrderBook,orderPriceMinTickSize,orderMinSize,volumeNum,liquidityNum,endDateIso,startDateIso,hasReviewedDates,volume24hr,volume1wk,volume1mo,volume1yr,clobTokenIds,umaBond,umaReward,volume24hrClob,volume1wkClob,volume1moClob,volume1yrClob,volumeClob,liquidityClob,acceptingOrders,negRisk,ready,funded,acceptingOrdersTimestamp,cyom,competitive,pagerDutyNotificationEnabled,approved,rewardsMinSize,rewardsMaxSpread,spread,oneDayPriceChange,oneWeekPriceChange,oneMonthPriceChange,lastTradePrice,bestBid,bestAsk,automaticallyActive,clearBookOnStart,manualActivation,negRiskOther,umaResolutionStatuses,pendingDeployment,deploying,rfqEnabled,holdingRewardsEnabled,feesEnabled,event_id,event_ticker,event_slug,event_title,event_description,event_startDate,event_creationDate,event_endDate,event_image,event_icon,event_active,event_closed,event_archived,event_new,event_featured,event_restricted,event_liquidity,event_volume,event_openInterest,event_createdAt,event_updatedAt,event_competitive,event_volume24hr,event_volume1wk,event_volume1mo,event_volume1yr,event_enableOrderBook,event_liquidityClob,event_commentCount,event_tags,event_cyom,event_showAllOutcomes,event_showMarketImages,event_enableNegRisk,event_automaticallyActive,event_negRiskAugmented,event_pendingDeployment,event_deploying,closedTime,umaEndDate,umaResolutionStatus,negRiskMarketID,negRiskRequestID,clobRewards,automaticallyResolved,seriesColor,event_resolutionSource,event_negRisk,event_negRiskMarketID,event_gmpChartMode,showGmpSeries,showGmpOutcome,oneHourPriceChange,event_sortBy,event_series,event_seriesSlug,volume24hrAmm,volume1wkAmm,volume1moAmm,volume1yrAmm,volumeAmm,liquidityAmm,oneYearPriceChange,gameStartTime,event_startTime,customLiveness,deployingTimestamp,event_featuredOrder,event_eventDate,outcomes_list,outcomePrices_list,clobTokenIds_list,umaResolutionStatuses_list,startDate_dt,endDate_dt,createdAt_dt,updatedAt_dt,acceptingOrdersTimestamp_dt,event_startDate_dt,event_endDate_dt,event_createdAt_dt,event_updatedAt_dt,volume_num,volumeNum_num,volumeClob_num,volume24hr_num,volume1wk_num,volume1mo_num,volume1yr_num,volume24hrClob_num,volume1wkClob_num,volume1moClob_num,volume1yrClob_num,liquidity_num,liquidityNum_num,liquidityClob_num,bestBid_num,bestAsk_num,lastTradePrice_num,spread_num,orderPriceMinTickSize_num,orderMinSize_num,competitive_num,rewardsMinSize_num,rewardsMaxSpread_num
119,516811,Will SpaceX have 200 or more launches in 2025?,0xc4e2af103f1f6e153cb07f9977b3cbdb5f04bb856a69036bbe8bd3...,will-spacex-have-200-or-more-launches-in-2025,https://www.faa.gov/data_research/commercial_space_data,2025-12-31T12:00:00Z,10590.2953,2024-12-30T15:45:04.881Z,https://polymarket-upload.s3.us-east-2.amazonaws.com/how...,https://polymarket-upload.s3.us-east-2.amazonaws.com/how...,"This market will resolve to ""Yes"" if SpaceX has 200 or m...","[""Yes"", ""No""]","[""0.034"", ""0.966""]",53998.851793,True,False,,2024-12-30T09:59:57.09901Z,2025-11-08T19:51:26.225403Z,False,False,0x91430CaD2d3975766499717fA0D66A78D814E5c5,False,0x2F5e3684cb1F318ec51b00Edba38d79Ac2c0aA9d,True,200 or more,6,0x2e006ec2cba16a9569c684f3c1d6034d733d6db47639320ecfb2c8...,True,0.001,5,5.399885e+04,10590.29530,2025-12-31,2024-12-30,True,62.701242,3.651883e+02,2.465012e+04,5.399885e+04,"[""380635031837051225328721248613354823043024287055111031...",500,5,62.701242,3.651883e+02,2.465012e+04,5.399885e+04,5.399885e+04,10590.29530,True,True,False,False,2024-12-30T15:43:54Z,False,0.821587,False,True,50,3.5,0.002,-0.0005,-0.0030,0.0015,0.033,0.033,0.035,True,True,False,False,[],False,False,False,False,False,16102,how-many-spacex-launches-in-2025,how-many-spacex-launches-in-2025,How many SpaceX launches in 2025?,This is a market on predict